# Test your IOL-AI 2026 script on Colab

A template to **test your competition script before you submit it**. It runs your model on real [Linguini](https://huggingface.co/datasets/facebook/linguini) problems — same format as the competition — so you can see what your script produces and check its answers.

The notebook has three parts:

- **Setup** and **Load the test problems** — Colab scaffolding, you don't touch it.
- **Your script** — your model + prompt + parsing. This is the part you experiment with and upload as `script.py`.
- **Check** — look at the output and score it (only possible here, where the answers are known).

**From this notebook to a real submission, only three things change:**

1. `MODEL_ID` points at `.` (your weights ship inside the submission repo) instead of a Hub name.
2. You read the platform's hidden test set at `/tmp/data/test.csv` instead of Linguini.
3. You write `submission.csv` instead of scoring here.

Set **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

## Setup

Installs the loader and pins `numpy`. Colab loads an older `numpy` at startup, so this cell **restarts the runtime once** after installing, so the new versions load cleanly. After it restarts, just **carry on with the cells below** (start again from "Load the test problems"). If it doesn't restart on its own: Runtime → Restart session, then continue.

In [ ]:
# Colab has internet, so we install the AWQ loader (gptqmodel) + deps. The offline
# sandbox installs nothing — autoawq is already there. numpy is pinned so the
# upgrade stays consistent.
!pip install -q -U transformers accelerate datasets sacrebleu gptqmodel "numpy==2.2.6"

# Colab loaded an older numpy at startup; the install just replaced it on disk, so we
# restart the runtime once to load the new one. After the restart, carry on with the
# cells below (re-running this one is fine too — it won't restart a second time).
import numpy, IPython
if numpy.__version__ != "2.2.6":
    print("Installed — restarting. When it's back, continue with the cells below.", flush=True)
    IPython.Application.instance().kernel.do_shutdown(True)

## Load the test problems

Colab scaffolding — you don't change this. We load 8 Linguini problems, each with `context`, `query`, and the reference `answer`. In a real submission the platform hands you the same `context` / `query` columns at `/tmp/data/test.csv` (without the answers — which is exactly why you test here first).

In [ ]:
from datasets import load_dataset

# Colab: Linguini (public, has reference answers).
# Submission: df = pd.read_csv("/tmp/data/test.csv", dtype=str).fillna("")   # same context/query columns
ds = load_dataset("facebook/linguini")
df = ds["test"].to_pandas().sample(8, random_state=0).reset_index(drop=True)
print(len(df), "problems |", list(df["task_type"].unique()))

## Your script

Everything from here to **Check** is your submission — the model, the prompt, and how you read the answers back. This is the part you experiment with and upload as `script.py`. It takes each problem's `context` + `query` from `df`, asks the model, and parses its answer.

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Free a model already on the GPU, so re-running this cell doesn't stack a second copy and run out of
# memory. (If you still hit "out of memory", do Runtime -> Restart session and run this cell just once.)
model = tok = None
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

MODEL_ID = "ritaberrada/iol-qwen25-14b-awq"   # <- your submission repo (in the sandbox this becomes ".")
MAX_NEW_TOKENS = 1536                          # room to reason; lower = faster but answers may get cut off

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
).eval()
print("loaded", MODEL_ID, "| VRAM", round(torch.cuda.max_memory_allocated() / 1e9, 1), "GB")

In [ ]:
import re

# The prompt: how you ask the model. This is the main thing you tweak to get better answers.
SYSTEM = (
    "You solve International Linguistics Olympiad problems by reasoning from the "
    "data given. Reason step by step. Then write a line that says exactly "
    "FINAL ANSWERS: and, below it, one answer per line in the order the items are "
    "asked -- the bare answer only, no numbering, no quotes, no extra text."
)

def parse_answers(text):
    """Keep only the lines after the last 'FINAL ANSWERS:' marker, one answer per line."""
    marker = list(re.finditer(r"(?im)^\s*final answers?\s*:?\s*$", text))
    if marker:
        text = text[marker[-1].end():]
    answers = []
    for line in text.splitlines():
        line = re.sub(r"^\s*\d+[.)]\s*", "", line).strip()   # drop "1. " / "2) " if the model adds it
        if line:
            answers.append(line)
    return answers

In [ ]:
# Answer every problem, in order.
results = []
for i, r in df.iterrows():
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{r['context'].strip()}\n\n{r['query'].strip()}"},
    ]
    ids = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    text = tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()
    answers = parse_answers(text)
    # A submission would instead collect {"id": r["id"], "pred": json.dumps(answers)} and write submission.csv.
    results.append({"query": r["query"], "reference": r["answer"], "raw": text, "pred": answers})
    print(f"[{i + 1}/{len(df)}] {len(answers)} answers", flush=True)

## Check your script (Colab only)

This is what you can't do in the sandbox. Because Linguini ships the reference answers, you can look at the raw output and score it the way the competition does.

In [ ]:
for i, res in enumerate(results, 1):
    print("=" * 72)
    print(f"PROBLEM {i}")
    print("=" * 72)
    print(res["query"][:300])
    print("\n--- RAW MODEL OUTPUT ---")
    print(res["raw"])
    print("\nPARSED ANSWERS:", res["pred"])
    print("REFERENCE:     ", res["reference"])
    print()

In [ ]:
import ast
from sacrebleu.metrics import CHRF
chrf = CHRF()

def gold_alts(reference):
    """Reference: a list of items, each a list of accepted alternatives."""
    v = ast.literal_eval(str(reference))
    return [[str(a) for a in it] if isinstance(it, (list, tuple)) else [str(it)] for it in v]

em = cf = n = 0
for res in results:
    gold = gold_alts(res["reference"])
    preds = (res["pred"] + [""] * len(gold))[:len(gold)]   # line up by position, as the scorer does
    for p, alts in zip(preds, gold):
        em += any(p.strip().lower() == a.strip().lower() for a in alts)
        cf += max(chrf.sentence_score(p, [a]).score for a in alts)
        n += 1

print(f"items scored: {n}")
print(f"exact match:  {em / n:.2f}")
print(f"chrF:         {cf / n:.1f}")